# Enhanced AAA Movie Revenue Prediction

This notebook expands upon `2_predict_aaa_movies.ipynb` by integrating richer data sources, advanced feature engineering, and state-of-the-art models.

## Setup
* Install additional libraries:
  ```bash
  pip install lightgbm catboost optuna featuretools networkx transformers sentencepiece kaggle pytrends
  ```
* Optional data downloads:
  * [The Numbers](https://www.the-numbers.com/market/) or [Box Office Mojo](https://www.boxofficemojo.com/) CSV exports for worldwide box office and marketing spend.
  * [Rotten Tomatoes Open Dataset](https://www.kaggle.com/datasets/dalziel/rotten-tomatoes-movies-and-critics-dataset) or [Metacritic data](https://www.kaggle.com/datasets/PromptCloudHQ/imdb-data) for critic and audience sentiment.
  * [Google Trends](https://trends.google.com/) via `pytrends` for pre-release search interest.
  * Social media buzz from Twitter API v2 or other sources.

Ensure that the original DuckDB database from previous notebooks is available at `data/movies.duckdb`.

In [ ]:
# Uncomment to install new dependencies in a fresh environment
%pip install lightgbm catboost optuna featuretools networkx transformers sentencepiece kaggle pytrends -q


In [ ]:
import duckdb, pandas as pd, numpy as np
import featuretools as ft
import networkx as nx
from transformers import AutoTokenizer, AutoModel
from pytrends.request import TrendReq
import lightgbm as lgb, catboost as cb
import optuna

con = duckdb.connect('data/movies.duckdb')
movies = con.execute('SELECT * FROM movies').df()
movies.head()


## External data integration
### Box office and marketing spend
Merge additional financial data from Box Office Mojo or The Numbers.


In [ ]:
# numbers = pd.read_csv('data/the_numbers.csv')  # download separately
# movies = movies.merge(numbers, on='primary_title', how='left')


### Critic and audience scores
Ingest Rotten Tomatoes or Metacritic scores to capture sentiment.


In [ ]:
# rt = pd.read_csv('data/rotten_tomatoes.csv')
# movies = movies.merge(rt, on='imdb_id', how='left')


### Social trend features
Use Google Trends to estimate pre-release buzz for each title.


In [ ]:
# pytrends = TrendReq()
# pytrends.build_payload([title], timeframe='2020-01-01 2020-12-31')
# interest = pytrends.interest_over_time()
# movies.loc[movies.primary_title == title, 'google_trends'] = interest[title].mean()


### Cast & crew networks
Model collaboration networks using `networkx` to quantify star power.


In [ ]:
# G = nx.Graph()
# for _, row in cast_df.iterrows():
#     G.add_edge(row['title_id'], row['person_id'])
# movies['degree_centrality'] = movies['tconst'].map(nx.degree_centrality(G))


### Plot summary embeddings
Extract semantic features from plot summaries using Transformers.


In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
# model = AutoModel.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
# def embed(text):
#     tokens = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
#     return model(**tokens).last_hidden_state.mean(dim=1).detach().numpy()
# movies['plot_embedding'] = movies['plot'].apply(embed)


### Automated feature generation
Leverage `featuretools` for deep feature synthesis.


In [ ]:
# es = ft.EntitySet.from_dataframe('movies', movies, index='tconst')
# feature_matrix, feature_defs = ft.dfs(entityset=es, target_entity='movies')
# movies = feature_matrix


## Modeling with hyperparameter tuning and ensembling
Train LightGBM and CatBoost models with Optuna for hyperparameter search and ensemble their predictions.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, r2_score

y = movies['revenue']
X = movies.drop(columns=['revenue'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'mape',
        'num_leaves': trial.suggest_int('num_leaves', 31, 256),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 0.3),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0)
    }
    dtrain = lgb.Dataset(X_train, label=y_train)
    gbm = lgb.train(params, dtrain, num_boost_round=100)
    preds = gbm.predict(X_test)
    return mean_absolute_percentage_error(y_test, preds)

# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=30)
# best_params = study.best_trial.params
# final_model = lgb.train({**best_params, 'objective': 'regression'}, dtrain)

# CatBoost and ensembling
# cat_model = cb.CatBoostRegressor(loss_function='MAPE', verbose=False)
# cat_model.fit(X_train, y_train)
# lgb_pred = final_model.predict(X_test)
# cat_pred = cat_model.predict(X_test)
# ensemble_pred = (lgb_pred + cat_pred) / 2
# print('MAPE:', mean_absolute_percentage_error(y_test, ensemble_pred))
# print('R^2:', r2_score(y_test, ensemble_pred))


## Next steps
* Log experiments with MLflow or Weights & Biases.
* Apply SHAP for model explainability.
* Deploy the trained model as a web service or batch scoring job.
